<a href="https://colab.research.google.com/github/lassieo/Yelp---Restaurants-in-New-Orleans/blob/main/Data_Cleaning_DSBA_6211_Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from typing import Dict

import kagglehub


def build_dataset_file_map(dataset_slug: str) -> tuple[str, Dict[str, str]]:
    """Download the Kaggle dataset and return a filename -> path mapping for all JSON files."""
    dataset_path = kagglehub.dataset_download(dataset_slug)
    json_files = sorted(
        file_name for file_name in os.listdir(dataset_path) if file_name.endswith(".json")
    )
    data_file_paths = {
        file_name: os.path.join(dataset_path, file_name)
        for file_name in json_files
    }
    return dataset_path, data_file_paths


DATASET_SLUG = "yelp-dataset/yelp-dataset"
DATASET_PATH, DATA_FILE_PATHS = build_dataset_file_map(DATASET_SLUG)

print(f"Dataset downloaded to: {DATASET_PATH}")
print(f"Discovered {len(DATA_FILE_PATHS)} JSON files:")
for file_name in DATA_FILE_PATHS:
    print(f"  - {file_name}")


Using Colab cache for faster access to the 'yelp-dataset' dataset.
Dataset downloaded to: /kaggle/input/yelp-dataset
Discovered 5 JSON files:
  - yelp_academic_dataset_business.json
  - yelp_academic_dataset_checkin.json
  - yelp_academic_dataset_review.json
  - yelp_academic_dataset_tip.json
  - yelp_academic_dataset_user.json


In [ ]:
import pandas as pd


def load_sample_json_frames(
    data_file_paths: dict[str, str],
    nrows: int = 10_000,
) -> dict[str, pd.DataFrame]:
    """Load a manageable sample from each Yelp JSON file."""
    sample_frames: dict[str, pd.DataFrame] = {}

    for file_name, file_path in data_file_paths.items():
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Required file not found: {file_path}")

        sample_frames[file_name] = pd.read_json(
            file_path,
            lines=True,
            nrows=nrows,
        )

    return sample_frames


SAMPLE_ROWS = 10_000
sample_dfs = load_sample_json_frames(DATA_FILE_PATHS, nrows=SAMPLE_ROWS)

print(f"Loaded {len(sample_dfs)} sample DataFrames ({SAMPLE_ROWS:,} rows max per file).")
print("Available tables:")
for file_name, df in sample_dfs.items():
    print(f"  - {file_name}: {df.shape}")


Loaded 5 sample DataFrames (10,000 rows max per file).
Available tables:
  - yelp_academic_dataset_business.json: (10000, 14)
  - yelp_academic_dataset_checkin.json: (10000, 2)
  - yelp_academic_dataset_review.json: (10000, 9)
  - yelp_academic_dataset_tip.json: (10000, 5)
  - yelp_academic_dataset_user.json: (10000, 22)


## Cleaning Sandbox

- Use a few small experiments to test messy columns before building the final pipeline.
- Focus on patterns that matter for CSV export: nested dictionaries, timestamp parsing, and list-like text fields.
- Keep these cells exploratory; the reusable logic lives later in the helper section.


In [ ]:
df_business = sample_dfs["yelp_academic_dataset_business.json"]

print("--- Sandbox: flatten nested business hours ---")
df_biz_test = df_business[["business_id", "name", "hours"]].copy().head(5)

hours_expanded = pd.json_normalize(
    df_biz_test["hours"].apply(lambda value: value if isinstance(value, dict) else {})
)

df_biz_clean = pd.concat(
    [df_biz_test.drop(columns="hours"), hours_expanded],
    axis=1,
)

display(df_biz_clean)


--- Sandbox: flatten nested business hours ---


,business_id,name,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,0:0-0:0,8:0-18:30,8:0-18:30,8:0-18:30,8:0-18:30,8:0-14:0,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,8:0-22:0,8:0-22:0,8:0-22:0,8:0-22:0,8:0-23:0,8:0-23:0,8:0-22:0
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,7:0-20:0,7:0-20:0,7:0-20:0,7:0-20:0,7:0-21:0,7:0-21:0,7:0-21:0
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,NaN,NaN,14:0-22:0,16:0-22:0,12:0-22:0,12:0-22:0,12:0-18:0


In [ ]:
df_checkin = sample_dfs["yelp_academic_dataset_checkin.json"]

print("\n--- Sandbox: explode check-in timestamps ---")
df_chk_test = df_checkin.head(2).copy()

df_chk_test["date"] = df_chk_test["date"].str.split(", ")
df_chk_clean = df_chk_test.explode("date")
df_chk_clean["date"] = pd.to_datetime(df_chk_clean["date"])

display(df_chk_clean.head(10))



--- Sandbox: explode check-in timestamps ---


,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,2020-03-13 21:10:56
0,---kPU91CF4Lq2-WlRu9Lw,2020-06-02 22:18:06
0,---kPU91CF4Lq2-WlRu9Lw,2020-07-24 22:42:27
0,---kPU91CF4Lq2-WlRu9Lw,2020-10-24 21:36:13
0,---kPU91CF4Lq2-WlRu9Lw,2020-12-09 21:23:33
0,---kPU91CF4Lq2-WlRu9Lw,2021-01-20 17:34:57
0,---kPU91CF4Lq2-WlRu9Lw,2021-04-30 21:02:03
0,---kPU91CF4Lq2-WlRu9Lw,2021-05-25 21:16:54
0,---kPU91CF4Lq2-WlRu9Lw,2021-08-06 21:08:08
0,---kPU91CF4Lq2-WlRu9Lw,2021-10-02 15:15:42


In [ ]:
df_user = sample_dfs["yelp_academic_dataset_user.json"]

print("\n--- Sandbox: normalize user date and elite fields ---")
df_usr_test = df_user[["user_id", "yelping_since", "elite"]].copy().head(5)

df_usr_test["yelping_since_clean"] = pd.to_datetime(df_usr_test["yelping_since"])
df_usr_test["elite_list"] = df_usr_test["elite"].apply(
    lambda value: value.split(",") if value else []
)

display(df_usr_test)
print("\nData types after cleaning:")
print(df_usr_test[["yelping_since_clean", "elite_list"]].dtypes)



--- Sandbox: normalize user date and elite fields ---


,user_id,yelping_since,elite,yelping_since_clean,elite_list
0,qVc8ODYU5SZjKXVBgXdI7w,2007-01-25 16:47:26,2007,2007-01-25 16:47:26,[2007]
1,j14WgRoU_-2ZE1aw1dXrJg,2009-01-25 04:35:42,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...",2009-01-25 04:35:42,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201..."
2,2WnXYQFK0hXEoTxPtV2zvg,2008-07-25 10:41:00,"2009,2010,2011,2012,2013",2008-07-25 10:41:00,"[2009, 2010, 2011, 2012, 2013]"
3,SZDeASXq7o05mMNLshsdIA,2005-11-29 04:38:33,"2009,2010,2011",2005-11-29 04:38:33,"[2009, 2010, 2011]"
4,hA5lMy-EnncsH4JoR-hFGQ,2007-01-05 19:40:59,,2007-01-05 19:40:59,[]



Data types after cleaning:
yelping_since_clean    datetime64[ns]
elite_list                     object
dtype: object


### Cleaning Sandbox Snapshot

- Quick before/after comparison of the three cleanup experiments above.
- Useful for explaining what changed without scrolling through every intermediate table.


In [ ]:
from IPython.display import HTML, display


def truncate_preview(value: object, limit: int = 45) -> str:
    text = str(value)
    return text if len(text) <= limit else text[:limit] + "..."


biz_orig_val = truncate_preview(df_business["hours"].iloc[1])
biz_clean_val = truncate_preview(df_biz_clean["Monday"].iloc[1])

chk_orig_val = truncate_preview(df_checkin["date"].iloc[0])
chk_clean_val = truncate_preview(df_chk_clean["date"].iloc[0])

usr_ys_orig_val = truncate_preview(df_user["yelping_since"].iloc[0])
usr_ys_clean_val = truncate_preview(df_usr_test["yelping_since_clean"].iloc[0])

usr_el_orig_val = df_user["elite"].iloc[0]
usr_el_orig_val = "'' (empty string)" if usr_el_orig_val == "" else truncate_preview(usr_el_orig_val)
usr_el_clean_val = truncate_preview(df_usr_test["elite_list"].iloc[0])

html = f"""
<style>
  .custom-table-container {{
    font-family: Arial, sans-serif;
    margin-top: 20px;
  }}
  .custom-table-container h3 {{
    color: #333;
  }}
  .custom-table {{
    width: 100%;
    border-collapse: collapse;
    text-align: left;
    box-shadow: 0 4px 8px rgba(0,0,0,0.1);
  }}
  .custom-table th, .custom-table td {{
    padding: 12px;
    border: 1px solid #ddd;
  }}
  .custom-table th {{
    color: white;
  }}
  .custom-table tbody tr:nth-child(even) {{
    background-color: #f9f9f9;
  }}
  .custom-table tbody tr:nth-child(odd) {{
    background-color: #ffffff;
  }}
  @media (prefers-color-scheme: dark) {{
    .custom-table-container h3 {{
      color: #f5f5f5;
    }}
    .custom-table th, .custom-table td {{
      border: 1px solid #444;
      color: #e0e0e0;
    }}
    .custom-table tbody tr:nth-child(even) {{
      background-color: #2a2a2a;
    }}
    .custom-table tbody tr:nth-child(odd) {{
      background-color: #1e1e1e;
    }}
  }}
</style>
<div class="custom-table-container">
    <h3>Cleaning Sandbox: Before vs. After</h3>
    <table class="custom-table">
        <thead>
            <tr style="background-color: #4CAF50;">
                <th>Dataset</th>
                <th>Original Column</th>
                <th>Type (Before)</th>
                <th>Example (Before)</th>
                <th style="background-color: #45a049;">Cleaned Column</th>
                <th style="background-color: #45a049;">Type (After)</th>
                <th style="background-color: #45a049;">Example (After)</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td><b>Business</b></td>
                <td>hours</td>
                <td style="color: #d9534f;">{df_business["hours"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{biz_orig_val}</td>
                <td>Monday <i>(flattened)</i></td>
                <td style="color: #5cb85c;">{df_biz_clean["Monday"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{biz_clean_val}</td>
            </tr>
            <tr>
                <td><b>Check-in</b></td>
                <td>date</td>
                <td style="color: #d9534f;">{df_checkin["date"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{chk_orig_val}</td>
                <td>date <i>(exploded)</i></td>
                <td style="color: #5cb85c;">{df_chk_clean["date"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{chk_clean_val}</td>
            </tr>
            <tr>
                <td><b>User</b></td>
                <td>yelping_since</td>
                <td style="color: #d9534f;">{df_user["yelping_since"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{usr_ys_orig_val}</td>
                <td>yelping_since_clean</td>
                <td style="color: #5cb85c;">{df_usr_test["yelping_since_clean"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{usr_ys_clean_val}</td>
            </tr>
            <tr>
                <td><b>User</b></td>
                <td>elite</td>
                <td style="color: #d9534f;">{df_user["elite"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{usr_el_orig_val}</td>
                <td>elite_list</td>
                <td style="color: #5cb85c;">{df_usr_test["elite_list"].dtype}</td>
                <td style="font-family: monospace; font-size: 0.9em;">{usr_el_clean_val}</td>
            </tr>
        </tbody>
    </table>
</div>
"""

display(HTML(html))


Dataset,Original Column,Type (Before),Example (Before),Cleaned Column,Type (After),Example (After)
Business,hours,object,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30',...",Monday (flattened),object,0:0-0:0
Check-in,date,object,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 202...",date (exploded),datetime64[ns],2020-03-13 21:10:56
User,yelping_since,object,2007-01-25 16:47:26,yelping_since_clean,datetime64[ns],2007-01-25 16:47:26
User,elite,object,2007,elite_list,object,['2007']


## Initial EDA

### 1. Business Data (`yelp_academic_dataset_business.json`)

- Core business metadata.
- Includes nested fields such as `attributes` and `hours`.
- This is the dataset used in the final New Orleans export pipeline.


In [ ]:
df_business = sample_dfs['yelp_academic_dataset_business.json']
display(df_business.head())
print("\n--- DataFrame Info ---")
df_business.info()
print("\n--- DataFrame Describe ---")
display(df_business.describe(include='all'))

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   10000 non-null  object 
 1   name          10000 non-null  object 
 2   address       10000 non-null  object 
 3   city          10000 non-null  object 
 4   state         10000 non-null  object 
 5   postal_code   10000 non-null  object 
 6   latitude      10000 non-null  float64
 7   longitude     10000 non-null  float64
 8   stars         10000 non-null  float64
 9   review_count  10000 non-null  int64  
 10  is_open       10000 non-null  int64  
 11  attributes    9140 non-null   object 
 12  categories    9994 non-null   object 
 13  hours         8492 non-null   object 
dtypes: float64(3), int64(2), object(9)
memory usage: 1.1+ MB

--- DataFrame Describe ---


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
count,10000,10000,10000,10000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,9140,9994,8492
unique,10000,8855,9499,563,16,1098,NaN,NaN,NaN,NaN,NaN,5673,7359,5284
top,EGITUFVLb87GRKapMtZg-w,McDonald's,,Philadelphia,PA,93101,NaN,NaN,NaN,NaN,NaN,{'BusinessAcceptsCreditCards': 'True'},"Restaurants, Pizza","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W..."
freq,1,54,323,980,2266,145,NaN,NaN,NaN,NaN,NaN,618,66,463
mean,NaN,NaN,NaN,NaN,NaN,NaN,36.687128,-89.414131,3.609800,46.963000,0.796700,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,5.907506,14.967018,0.974828,122.302955,0.402474,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,27.584300,-120.095137,1.000000,5.000000,0.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,32.192505,-90.358702,3.000000,8.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,38.767180,-86.126620,4.000000,15.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,39.953403,-75.411271,4.500000,39.000000,1.000000,NaN,NaN,NaN


### 2. Review Data (`yelp_academic_dataset_review.json`)

- Long-form review text with linked `user_id` and `business_id`.
- Useful for text mining or sentiment work later, but not part of the export pipeline here.


In [ ]:
df_review = sample_dfs['yelp_academic_dataset_review.json']
display(df_review.head())
print("\n--- DataFrame Info ---")
df_review.info()
print("\n--- DataFrame Describe ---")
display(df_review.describe(include='all'))

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   review_id    10000 non-null  object        
 1   user_id      10000 non-null  object        
 2   business_id  10000 non-null  object        
 3   stars        10000 non-null  int64         
 4   useful       10000 non-null  int64         
 5   funny        10000 non-null  int64         
 6   cool         10000 non-null  int64         
 7   text         10000 non-null  object        
 8   date         10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 703.3+ KB

--- DataFrame Describe ---


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
count,10000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000,10000
unique,10000,9472,3930,NaN,NaN,NaN,NaN,10000,NaN
top,HZUfjEGHLihvDZFlurjxpQ,n-lBS02-3yvlY5Q91mmwDA,GBTPC53ZrG1ZBY3DT8Mbcw,NaN,NaN,NaN,NaN,I was in the market for a car and went here to...,NaN
freq,1,6,85,NaN,NaN,NaN,NaN,1,NaN
mean,NaN,NaN,NaN,3.854300,0.889100,0.246500,0.335500,NaN,2015-04-17 08:27:40.820000
min,NaN,NaN,NaN,1.000000,0.000000,0.000000,0.000000,NaN,2005-03-01 17:47:15
25%,NaN,NaN,NaN,3.000000,0.000000,0.000000,0.000000,NaN,2013-11-14 11:16:35.500000
50%,NaN,NaN,NaN,4.000000,0.000000,0.000000,0.000000,NaN,2015-09-09 23:20:24
75%,NaN,NaN,NaN,5.000000,1.000000,0.000000,0.000000,NaN,2017-03-27 02:25:32.500000
max,NaN,NaN,NaN,5.000000,91.000000,26.000000,44.000000,NaN,2018-10-04 18:22:35


### 3. User Data (`yelp_academic_dataset_user.json`)

- User profile metadata, activity counts, and date-like fields.
- Helpful for testing type conversion and list-style cleanup.


In [ ]:
df_user = sample_dfs['yelp_academic_dataset_user.json']
display(df_user.head())
print("\n--- DataFrame Info ---")
df_user.info()
print("\n--- DataFrame Describe ---")
display(df_user.describe(include='all'))

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             10000 non-null  object 
 1   name                10000 non-null  object 
 2   review_count        10000 non-null  int64  
 3   yelping_since       10000 non-null  object 
 4   useful              10000 non-null  int64  
 5   funny               10000 non-null  int64  
 6   cool                10000 non-null  int64  
 7   elite               10000 non-null  object 
 8   friends             10000 non-null  object 
 9   fans                10000 non-null  int64  
 10  average_stars       10000 non-null  float64
 11  compliment_hot      10000 non-null  int64  
 12  compliment_more     10000 non-null  int64  
 13  compliment_profile  10000 non-null  int64  
 14  compliment_cute     10000 non-null  int64  
 15  compliment_list     10000 non-

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
count,10000,10000,10000.00000,10000,10000.000000,10000.000000,10000.000000,10000,10000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
unique,10000,3146,NaN,10000,NaN,NaN,NaN,484,10000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,g6Fj4RZoLtDN_huxFgCJ6g,Michael,NaN,2005-08-24 03:24:09,NaN,NaN,NaN,,"UUoU2s6BpIwu3nOuPMizfg, KVmhGRWJjWouy45gebVzLw...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,106,NaN,1,NaN,NaN,NaN,5067,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,278.61810,NaN,911.666300,465.440600,606.360300,NaN,NaN,32.559700,...,8.324200,6.862600,5.405000,3.415300,38.890800,95.107200,85.805900,85.805900,33.117200,23.826100
std,NaN,NaN,502.63081,NaN,3695.356472,2292.975591,3071.287502,NaN,NaN,162.946065,...,64.614323,97.454004,38.826959,39.377848,186.300067,606.325624,451.382152,451.382152,184.173485,245.570463
min,NaN,NaN,1.00000,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,34.75000,NaN,45.750000,10.000000,14.000000,NaN,NaN,2.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
50%,NaN,NaN,115.00000,NaN,176.000000,55.000000,68.000000,NaN,NaN,7.000000,...,2.000000,0.000000,0.000000,0.000000,5.000000,6.000000,6.000000,6.000000,3.000000,1.000000
75%,NaN,NaN,323.00000,NaN,628.000000,239.000000,299.000000,NaN,NaN,25.000000,...,5.000000,2.000000,2.000000,1.000000,19.000000,28.000000,33.000000,33.000000,16.000000,4.000000


### 4. Check-in Data (`yelp_academic_dataset_checkin.json`)

- Business-level timestamp activity stored as comma-separated strings.
- A good example of why exploratory cleanup is necessary before analysis.


In [ ]:
df_checkin = sample_dfs['yelp_academic_dataset_checkin.json']
display(df_checkin.head())
print("\n--- DataFrame Info ---")
df_checkin.info()
print("\n--- DataFrame Describe ---")
display(df_checkin.describe(include='all'))

,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22"
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012..."
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014..."



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   business_id  10000 non-null  object
 1   date         10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB

--- DataFrame Describe ---


,business_id,date
count,10000,10000
unique,10000,10000
top,3mFWy3lthYu8mBJsNavL7Q,"2010-04-16 00:29:14, 2011-01-19 23:41:34, 2011..."
freq,1,1


### 5. Tip Data (`yelp_academic_dataset_tip.json`)

- Short-form user tips tied to businesses.
- Useful for lightweight text analysis and behavior patterns.


In [ ]:
df_tip = sample_dfs['yelp_academic_dataset_tip.json']
display(df_tip.head())
print("\n--- DataFrame Info ---")
df_tip.info()
print("\n--- DataFrame Describe ---")
display(df_tip.describe(include='all'))

,user_id,business_id,text,date,compliment_count
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21,0
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10,0
2,-copOvldyKh1qr-vzkDEvw,MYoRNLb5chwjQe3c_k37Gg,It's open even when you think it isn't,2013-08-18 00:56:08,0
3,FjMQVZjSqY8syIO-53KFKw,hV-bABTK-glh5wj31ps_Jw,Very decent fried chicken,2017-06-27 23:05:38,0
4,ld0AperBXk1h6UbqmM80zw,_uN0OudeJ3Zl_tf6nxg5ww,Appetizers.. platter special for lunch,2012-10-06 19:43:09,0



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           10000 non-null  object        
 1   business_id       10000 non-null  object        
 2   text              10000 non-null  object        
 3   date              10000 non-null  datetime64[ns]
 4   compliment_count  10000 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 390.8+ KB

--- DataFrame Describe ---


,user_id,business_id,text,date,compliment_count
count,10000,10000,10000,10000,10000.000000
unique,6227,7900,9935,NaN,NaN
top,fCvMnJU1Z-XhAjKg99wK3Q,-QI8Qi8XWH3D8y8ethnajA,Love this place!,NaN,NaN
freq,44,13,5,NaN,NaN
mean,NaN,NaN,NaN,2013-12-30 05:01:39.893100032,0.013400
min,NaN,NaN,NaN,2009-04-24 04:59:59,0.000000
25%,NaN,NaN,NaN,2012-04-04 00:34:14.249999872,0.000000
50%,NaN,NaN,NaN,2013-12-01 22:26:48.500000,0.000000
75%,NaN,NaN,NaN,2015-09-07 00:16:12.249999872,0.000000
max,NaN,NaN,NaN,2018-05-04 22:32:49,3.000000


## Reusable Helpers

- Small utility functions used later in the notebook.
- Kept separate from the sandbox so the final pipeline is easier to rerun and maintain.


In [ ]:
#@title Data Type Visualizer Helper { display-mode: "form" }
from __future__ import annotations

from html import escape
from typing import Final

import pandas as pd
from IPython.display import HTML, display

DTYPE_COLORS: Final[dict[str, str]] = {
    "object": "#e74c3c",
    "int64": "#3498db",
    "float64": "#2ecc71",
    "bool": "#9b59b6",
    "datetime64[ns]": "#f39c12",
}


def render_dtype_chips(df: pd.DataFrame, title: str = "Data Types") -> None:
    """Render DataFrame columns as color-coded HTML chips by dtype."""
    if df.empty and len(df.columns) == 0:
        raise ValueError("DataFrame has no columns to render.")

    chips: list[str] = []
    for column_name, dtype in df.dtypes.items():
        dtype_str = str(dtype)
        color = DTYPE_COLORS.get(dtype_str, "#7f8c8d")
        chips.append(
            f"""
            <span style="
                display: inline-block;
                margin: 6px 8px 6px 0;
                padding: 8px 14px;
                border-radius: 999px;
                background: {color};
                color: white;
                font-family: Arial, sans-serif;
                font-size: 16px;
                font-weight: 700;
                line-height: 1.2;
            ">
                {escape(str(column_name))} <span style="font-weight: 500; opacity: 0.9;">({escape(dtype_str)})</span>
            </span>
            """
        )

    html = f"""
    <style>
      .chip-container {{
        padding: 18px;
        border-radius: 14px;
        margin: 12px 0 20px 0;
        background: #f8f9fa;
        border: 1px solid #e0e0e0;
      }}
      .chip-title {{
        font-family: Arial, sans-serif;
        font-size: 24px;
        font-weight: 800;
        color: #202124;
        margin-bottom: 14px;
      }}
      @media (prefers-color-scheme: dark) {{
        .chip-container {{
          background: #111;
          border: 1px solid #333;
        }}
        .chip-title {{
          color: #f5f5f5;
        }}
      }}
    </style>
    <div class="chip-container">
        <div class="chip-title">{escape(title)}</div>
        <div>{''.join(chips)}</div>
    </div>
    """
    display(HTML(html))


In [ ]:
from __future__ import annotations

import pandas as pd


def clean_business_for_new_orleans_csv(df: pd.DataFrame) -> pd.DataFrame:
    """Clean Yelp business data for New Orleans CSV export."""
    required_columns = {
        "business_id",
        "name",
        "city",
        "state",
        "categories",
        "attributes",
        "hours",
    }
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    cleaned = df.copy()
    cleaned["city"] = cleaned["city"].astype(str).str.strip()
    cleaned["state"] = cleaned["state"].astype(str).str.strip()

    cleaned = cleaned.loc[cleaned["city"].str.lower() == "new orleans"].copy()
    if cleaned.empty:
        raise ValueError("No rows found for city == 'New Orleans'.")

    cleaned["categories"] = cleaned["categories"].apply(
        lambda value: ", ".join(value) if isinstance(value, list) else value
    )

    cleaned = cleaned.drop(columns=["attributes", "hours"], errors="ignore")
    cleaned = cleaned.drop_duplicates(subset="business_id")

    sort_columns = [col for col in ["stars", "review_count", "name"] if col in cleaned.columns]
    ascending_flags = [False, False, True][: len(sort_columns)]
    if sort_columns:
        cleaned = cleaned.sort_values(by=sort_columns, ascending=ascending_flags)

    return cleaned.reset_index(drop=True)


## Final Pipeline: New Orleans Business Export

- Reuse the helper functions above.
- Clean the business table for a focused New Orleans export.
- Run a few sanity checks, then write the CSV.


In [ ]:
df_business = sample_dfs["yelp_academic_dataset_business.json"]

print("--- Raw business data info ---")
df_business.info()

render_dtype_chips(df_business, title="Business Data Types — Raw")

business_clean = clean_business_for_new_orleans_csv(df_business)

render_dtype_chips(business_clean, title="Business Data Types — Cleaned")

print("\n--- Sanity checks ---")
print(f"Shape: {business_clean.shape}")
print(f"Unique cities:\n{business_clean['city'].value_counts().head()}")
print(f"Duplicate business_id values: {business_clean.duplicated(subset='business_id').sum()}")
display(business_clean.head())

business_clean.to_csv("new_orleans_business_clean.csv", index=False)
print("\nExported: new_orleans_business_clean.csv")


--- Raw business data info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   10000 non-null  object 
 1   name          10000 non-null  object 
 2   address       10000 non-null  object 
 3   city          10000 non-null  object 
 4   state         10000 non-null  object 
 5   postal_code   10000 non-null  object 
 6   latitude      10000 non-null  float64
 7   longitude     10000 non-null  float64
 8   stars         10000 non-null  float64
 9   review_count  10000 non-null  int64  
 10  is_open       10000 non-null  int64  
 11  attributes    9140 non-null   object 
 12  categories    9994 non-null   object 
 13  hours         8492 non-null   object 
dtypes: float64(3), int64(2), object(9)
memory usage: 1.1+ MB



--- Sanity checks ---
Shape: (440, 12)
Unique cities:
city
New Orleans    440
Name: count, dtype: int64
Duplicate business_id values: 0


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories
0,0yGH62WGQy_W3JoR6L5Kew,Ice Cream 504,2511 Jena St,New Orleans,LA,70115,29.935437,-90.104024,5.0,139,1,"Ice Cream & Frozen Yogurt, Food"
1,yAEKaEvsCNxOYv1M1kiQhg,Olive,339 Carondelet St,New Orleans,LA,70112,29.951080,-90.071576,5.0,76,1,"Lebanese, Halal, Restaurants, Breakfast & Brun..."
2,yIy70vHKF_L3Gy1cu2Wqwg,"Laurel Oak, Southern Brasserie",535 Gravier St,New Orleans,LA,70130,29.951021,-90.068527,5.0,57,1,"French, Restaurants, Southern"
3,aZZ5y7jcbaroGCal4MDxVg,Le Monde Creole,606 Royal St,New Orleans,LA,70130,29.957381,-90.065306,5.0,56,1,"Tours, Hotels & Travel"
4,VeM4d4GuAgLnghpCwlH72A,Angie Nola Maids,,New Orleans,LA,70119,29.979811,-90.079349,5.0,53,1,"Office Cleaning, Window Washing, Home Cleaning..."



Exported: new_orleans_business_clean.csv


## Data Cleaning + EDA: Merged Dataset

**Scope (W. Moore):** Merge all four data sources on `business_id`, filter to open New Orleans restaurants, add `success` and `tourist_zone` variables, validate data quality, and export the shared team dataset.

This section loads **full files** (not the 10 K sample) using the same `DATA_FILE_PATHS` dict built in the setup cell above.

### Step 1 — Load & Filter Business Data (full file)

- Keep only open restaurants (`is_open == 1`) in New Orleans.
- Require 'Restaurants' in categories.
- Retain `review_count` as a feature.

In [ ]:
import pandas as pd

biz_path = DATA_FILE_PATHS['yelp_academic_dataset_business.json']
df_biz_full = pd.read_json(biz_path, lines=True)

# Filter: New Orleans, open, has 'Restaurants' in categories
mask = (
    (df_biz_full['city'].str.strip().str.lower() == 'new orleans')
    & (df_biz_full['is_open'] == 1)
    & (df_biz_full['categories'].str.contains('Restaurants', na=False))
)
open_restaurants = df_biz_full[mask][
    ['business_id', 'name', 'stars', 'review_count', 'postal_code',
     'latitude', 'longitude', 'is_open']
].copy().reset_index(drop=True)

n_open = len(open_restaurants)
print(f'Open New Orleans restaurants: {n_open:,}')
print(f'Duplicate business_id: {open_restaurants.duplicated(subset="business_id").sum()}')
display(open_restaurants.head())


### Step 2 — Aggregate Review Data

The review file is ~5 GB, so it is read in chunks of 100 K rows.
Only reviews matching our open-restaurant `business_id` set are kept.
Aggregated columns: `avg_review_stars`, `review_text_count`.

In [ ]:
review_path = DATA_FILE_PATHS['yelp_academic_dataset_review.json']
target_ids = set(open_restaurants['business_id'])

chunks = []
for chunk in pd.read_json(review_path, lines=True, chunksize=100_000):
    filtered = chunk[chunk['business_id'].isin(target_ids)]
    if not filtered.empty:
        chunks.append(filtered[['business_id', 'stars']])

df_reviews = pd.concat(chunks, ignore_index=True)
review_agg = (
    df_reviews
    .groupby('business_id', as_index=False)
    .agg(avg_review_stars=('stars', 'mean'), review_text_count=('stars', 'count'))
)
review_agg['avg_review_stars'] = review_agg['avg_review_stars'].round(3)

print(f'Review rows matched: {len(df_reviews):,}')
print(f'Businesses with review data: {len(review_agg):,}')
display(review_agg.head())


### Step 3 — Aggregate Tip Data

Tips are short user comments with a `compliment_count`.
Aggregated columns: `tip_count`, `avg_tip_compliment_count`.

In [ ]:
tip_path = DATA_FILE_PATHS['yelp_academic_dataset_tip.json']
df_tip_full = pd.read_json(tip_path, lines=True)
df_tip_nola = df_tip_full[df_tip_full['business_id'].isin(target_ids)].copy()

tip_agg = (
    df_tip_nola
    .groupby('business_id', as_index=False)
    .agg(
        tip_count=('business_id', 'count'),
        avg_tip_compliment_count=('compliment_count', 'mean'),
    )
)
tip_agg['avg_tip_compliment_count'] = tip_agg['avg_tip_compliment_count'].round(3)

print(f'Businesses with tip data: {len(tip_agg):,}')
display(tip_agg.head())


### Step 4 — Aggregate Check-in Data

Check-in records store timestamps as a single comma-separated string.
Split and count to get total `checkin_count` per business.

In [ ]:
checkin_path = DATA_FILE_PATHS['yelp_academic_dataset_checkin.json']
df_chk_full = pd.read_json(checkin_path, lines=True)
df_chk_nola = df_chk_full[df_chk_full['business_id'].isin(target_ids)].copy()

df_chk_nola['checkin_count'] = df_chk_nola['date'].str.split(',').str.len()
checkin_agg = df_chk_nola[['business_id', 'checkin_count']].copy()

print(f'Businesses with check-in data: {len(checkin_agg):,}')
display(checkin_agg.head())


### Step 5 — Merge All Sources on `business_id`

Left join from the business base so no businesses are lost.
Restaurants with no reviews, tips, or check-ins get `NaN` for those columns.

In [ ]:
merged = (
    open_restaurants
    .merge(review_agg,  on='business_id', how='left')
    .merge(tip_agg,     on='business_id', how='left')
    .merge(checkin_agg, on='business_id', how='left')
)

# ── Validate: no businesses lost ──────────────────────────────────────
assert len(merged) == n_open, (
    f'Row count changed after merge: {n_open} → {len(merged)}'
)
print(f'Merge validation passed: {n_open:,} businesses before and after merge.')
print(f'Shape: {merged.shape}')
display(merged.head())


### Step 6 — Add `success` Variable

Binary target defined per the project's `Problem_Statements.md`:

| Stars | Label |
|-------|-------|
| ≥ 4.0 | 1 (high success) |
| < 3.5 | 0 (low success) |
| 3.5 – 3.99 | *excluded* (gray zone) |

In [ ]:
def assign_success(stars: float):
    if stars >= 4.0:
        return 1
    elif stars < 3.5:
        return 0
    return None  # gray zone — excluded

merged['success'] = merged['stars'].apply(assign_success)
n_before = len(merged)
merged = merged.dropna(subset=['success']).copy()
merged['success'] = merged['success'].astype(int)
n_after = len(merged)

print(f'Rows before gray-zone exclusion: {n_before:,}')
print(f'Rows after  gray-zone exclusion: {n_after:,}')
print(f'Excluded (gray zone):            {n_before - n_after:,}')
print()
print('Success distribution:')
print(merged['success'].value_counts().rename({1: 'High (≥4.0 stars)', 0: 'Low (<3.5 stars)'}))


### Step 7 — Add `tourist_zone` Variable

Binary flag based on New Orleans ZIP codes associated with high tourist traffic:

| ZIP | Area |
|-----|------|
| 70112 | French Quarter / CBD |
| 70113 | CBD / Lower Garden District |
| 70116 | French Quarter / Marigny |
| 70117 | Marigny / Bywater |
| 70130 | Warehouse / Arts District |

All other ZIP codes → `tourist_zone = 0`.

In [ ]:
TOURIST_ZIPS = {'70112', '70113', '70116', '70117', '70130'}

merged['tourist_zone'] = (
    merged['postal_code']
    .astype(str)
    .str.strip()
    .apply(lambda z: 1 if z in TOURIST_ZIPS else 0)
)

print('Tourist zone distribution:')
print(merged['tourist_zone'].value_counts().rename({1: 'Tourist zone', 0: 'Non-tourist'}))
print()
print('Success rate by tourist zone:')
print(merged.groupby('tourist_zone')['success'].mean().rename({0: 'Non-tourist', 1: 'Tourist zone'}))


### Step 8 — Data Quality Checks

- Duplicate `business_id` check
- Missing value summary
- Distribution of key numeric columns
- Confirm `review_count` is present

In [ ]:
import matplotlib.pyplot as plt

# 1. Duplicate check
dup_count = merged.duplicated(subset='business_id').sum()
assert dup_count == 0, f'Found {dup_count} duplicate business_id values!'
print(f'Duplicate business_id: {dup_count}  ✓')

# 2. review_count present
assert 'review_count' in merged.columns, 'review_count missing from dataset!'
print(f'review_count column present  ✓')

# 3. Missing values
print('\nMissing values per column:')
missing = merged.isnull().sum()
display(missing[missing > 0].to_frame('missing_count').assign(
    pct=lambda d: (d['missing_count'] / len(merged) * 100).round(1)
))
if missing.sum() == 0:
    print('  No missing values.')

# 4. Descriptive statistics
print('\nDescriptive statistics:')
display(merged[['stars', 'review_count', 'avg_review_stars',
                'tip_count', 'checkin_count', 'success', 'tourist_zone']].describe().round(2))

# 5. Stars distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
merged['stars'].hist(bins=9, ax=axes[0], edgecolor='white', color='steelblue')
axes[0].set_title('Star Rating Distribution')
axes[0].set_xlabel('Stars')
axes[0].set_ylabel('Count')

success_counts = merged['success'].value_counts().rename({1: 'High (≥4.0)', 0: 'Low (<3.5)'})
success_counts.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_title('Success Label Distribution')
axes[1].set_xlabel('Success')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


### Step 9 — Export Shared Dataset

Export `cleaned_restaurant_dataset.csv` for the team.
This file is the single source of truth for downstream modeling and analysis.

In [ ]:
output_cols = [
    'business_id', 'name', 'stars', 'review_count', 'postal_code',
    'latitude', 'longitude',
    'avg_review_stars', 'review_text_count',
    'tip_count', 'avg_tip_compliment_count',
    'checkin_count',
    'success', 'tourist_zone',
]
final_df = merged[output_cols].copy()

output_path = 'cleaned_restaurant_dataset.csv'
final_df.to_csv(output_path, index=False)

print(f'Exported: {output_path}')
print(f'Shape:    {final_df.shape[0]:,} rows  x  {final_df.shape[1]} columns')
print(f'Columns:  {list(final_df.columns)}')
display(final_df.head())


---

## What does a restaurant's review timeline actually look like?

The merged dataset tells us *how many* reviews each restaurant has, but not *when* they arrived or *what* people said. That matters — a restaurant with 200 reviews spread evenly across five years tells a very different story than one that got 200 reviews in a single summer.

This section re-reads the raw review file, keeps the full timestamp and text for every review tied to an open New Orleans restaurant, and exports a clean copy so the team can run time-based analysis and sentiment work downstream.

> **Christian — this is your handoff.** The export below gives you clean review text with usable datetime objects. No parsing surprises, no timezone guessing.

In [ ]:
# ── Step 10: Build the cleaned review dataset with timestamps ────────────────
# The review file is ~5 GB. Same chunked-read strategy as Step 2,
# but this time we keep: business_id, stars, date, text.

review_path = DATA_FILE_PATHS['yelp_academic_dataset_review.json']
target_ids  = set(open_restaurants['business_id'])

review_chunks = []
for chunk in pd.read_json(review_path, lines=True, chunksize=100_000):
    matched = chunk[chunk['business_id'].isin(target_ids)]
    if not matched.empty:
        review_chunks.append(matched[['business_id', 'stars', 'date', 'text']])

df_reviews_full = pd.concat(review_chunks, ignore_index=True)

# Convert date strings to proper datetime objects
df_reviews_full['date'] = pd.to_datetime(df_reviews_full['date'], errors='coerce')

# ── Quality gate ─────────────────────────────────────────────────────────────
nat_count  = df_reviews_full['date'].isna().sum()
empty_text = df_reviews_full['text'].isna().sum() + (df_reviews_full['text'].str.strip() == '').sum()

print(f"Total reviews matched:   {len(df_reviews_full):,}")
print(f"Date parsing failures:   {nat_count:,}")
print(f"Empty/null text rows:    {empty_text:,}")
print(f"Date range:              {df_reviews_full['date'].min():%Y-%m-%d}  to  {df_reviews_full['date'].max():%Y-%m-%d}")

# Drop any rows with bad dates or empty text
df_reviews_full = df_reviews_full.dropna(subset=['date', 'text'])
df_reviews_full = df_reviews_full[df_reviews_full['text'].str.strip() != ''].copy()

# ── The skew story (guardrail: always report median alongside mean) ──────────
reviews_per_biz = df_reviews_full.groupby('business_id').size()
print(f"\nReviews per restaurant:")
print(f"  Median: {reviews_per_biz.median():.0f}")
print(f"  Mean:   {reviews_per_biz.mean():.0f}")
print(f"  Max:    {reviews_per_biz.max():,}")
print(f"\n  That gap between median and mean is the skew.")
print(f"  A handful of heavily-reviewed restaurants — mostly in the tourist zone (ZIP 70130) — pull the average up.")

display(df_reviews_full.head())

In [ ]:
# ── Export for Christian ─────────────────────────────────────────────────────
review_export_path = 'cleaned_reviews_with_text.csv'
df_reviews_full.to_csv(review_export_path, index=False)

print(f"Exported: {review_export_path}")
print(f"Shape:    {df_reviews_full.shape[0]:,} rows  x  {df_reviews_full.shape[1]} columns")
print(f"Columns:  {list(df_reviews_full.columns)}")
print(f"\nChristian: this file has clean datetime objects and review text. No further parsing needed.")

## When did New Orleans restaurants get their reviews?

Not all months are created equal. Tourism seasons, holidays, and one very specific global event reshape how many reviews restaurants collect each month. Before we can talk about any of that, we need the raw monthly signal — total reviews per month, citywide and per restaurant.

This is the foundation for the pandemic analysis that follows.

In [ ]:
# ── Step 11: Monthly review volume time series ──────────────────────────────
df_reviews_full['review_month'] = df_reviews_full['date'].dt.to_period('M')

# Per-restaurant monthly counts
monthly_by_biz = (
    df_reviews_full
    .groupby(['business_id', 'review_month'])
    .size()
    .reset_index(name='monthly_review_count')
)

# City-level monthly totals
monthly_city = (
    df_reviews_full
    .groupby('review_month')
    .size()
    .reset_index(name='total_reviews')
)
monthly_city['review_month_dt'] = monthly_city['review_month'].dt.to_timestamp()

print(f"Monthly time series: {len(monthly_city)} months, {monthly_city['review_month'].min()} to {monthly_city['review_month'].max()}")
print(f"Per-restaurant rows: {len(monthly_by_biz):,}")
display(monthly_city.head())

In [ ]:
# ── The monthly pulse — where's the cliff? ───────────────────────────────────
# Tourist zone vs. non-tourist split (guardrail: always check this before
# reporting a city-wide finding).

# Tag each review with tourist_zone from the merged dataset
tz_lookup = merged[['business_id', 'tourist_zone']].drop_duplicates()
reviews_tagged = df_reviews_full.merge(tz_lookup, on='business_id', how='left')

monthly_by_zone = (
    reviews_tagged
    .groupby([reviews_tagged['date'].dt.to_period('M'), 'tourist_zone'])
    .size()
    .reset_index(name='review_count')
)
monthly_by_zone.columns = ['review_month', 'tourist_zone', 'review_count']
monthly_by_zone['month_dt'] = monthly_by_zone['review_month'].dt.to_timestamp()

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: city-wide
axes[0].plot(monthly_city['review_month_dt'], monthly_city['total_reviews'],
             color='steelblue', linewidth=1.5)
axes[0].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.7, label='COVID shock begins')
axes[0].set_title('Monthly Review Volume — All Open New Orleans Restaurants', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Reviews per month')
axes[0].legend()

# Bottom: tourist vs non-tourist
for tz, label, color in [(1, 'Tourist zone', '#f39c12'), (0, 'Non-tourist', '#3498db')]:
    subset = monthly_by_zone[monthly_by_zone['tourist_zone'] == tz]
    axes[1].plot(subset['month_dt'], subset['review_count'],
                 label=label, color=color, linewidth=1.5)
axes[1].axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Tourist Zone vs. Non-Tourist — Same Timeline, Different Recovery', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Reviews per month')
axes[1].set_xlabel('Month')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Export monthly time series ───────────────────────────────────────────────
monthly_export = monthly_by_biz.copy()
monthly_export['review_month'] = monthly_export['review_month'].astype(str)
monthly_export.to_csv('monthly_review_volume.csv', index=False)

print(f"Exported: monthly_review_volume.csv")
print(f"Shape:    {len(monthly_export):,} rows  x  {monthly_export.shape[1]} columns")

## How hard did COVID hit — and did the tourist zone recover differently?

This is a contained analysis section. The project's main event is predicting star ratings from restaurant characteristics — but review volume during a pandemic tells us something important about which restaurants kept their audience and which went quiet.

Three fixed windows:
- **Pre-COVID** (Jan 2018 – Feb 2020): What did normal look like?
- **Shock** (Mar 2020 – Jun 2020): How far did volume fall?
- **Recovery** (Jul 2020 onward): How quickly did it come back — and for whom?

In [ ]:
# ── Step 12: Pandemic period analysis ────────────────────────────────────────

# Tag each month with its pandemic period
def assign_period(month_dt):
    if month_dt < pd.Timestamp('2018-01-01'):
        return 'Before window'
    elif month_dt < pd.Timestamp('2020-03-01'):
        return 'Pre-COVID'
    elif month_dt < pd.Timestamp('2020-07-01'):
        return 'Shock'
    else:
        return 'Recovery'

monthly_city['period'] = monthly_city['review_month_dt'].apply(assign_period)

# Focus on the three analysis windows
analysis = monthly_city[monthly_city['period'].isin(['Pre-COVID', 'Shock', 'Recovery'])].copy()

# Baseline: median and mean monthly reviews in Pre-COVID window
pre_covid = analysis[analysis['period'] == 'Pre-COVID']['total_reviews']
baseline_median = pre_covid.median()
baseline_mean   = pre_covid.mean()

shock = analysis[analysis['period'] == 'Shock']['total_reviews']
shock_median = shock.median()
shock_mean   = shock.mean()

recovery = analysis[analysis['period'] == 'Recovery']['total_reviews']
recovery_median = recovery.median()
recovery_mean   = recovery.mean()

pct_drop_median = (1 - shock_median / baseline_median) * 100
pct_drop_mean   = (1 - shock_mean / baseline_mean) * 100

print("Pandemic Impact on Monthly Review Volume")
print("=" * 50)
print(f"{'Period':<15} {'Median':>10} {'Mean':>10}")
print("-" * 50)
print(f"{'Pre-COVID':<15} {baseline_median:>10.0f} {baseline_mean:>10.0f}")
print(f"{'Shock':<15} {shock_median:>10.0f} {shock_mean:>10.0f}")
print(f"{'Recovery':<15} {recovery_median:>10.0f} {recovery_mean:>10.0f}")
print("-" * 50)
print(f"Shock drop:     {pct_drop_median:.1f}% (median)  /  {pct_drop_mean:.1f}% (mean)")
print(f"Recovery vs baseline: {(recovery_median/baseline_median*100):.1f}% of pre-COVID median")

In [ ]:
# ── Period comparison — bar chart ────────────────────────────────────────────
period_summary = (
    analysis
    .groupby('period')['total_reviews']
    .agg(['median', 'mean'])
    .reindex(['Pre-COVID', 'Shock', 'Recovery'])
)

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(period_summary))
width = 0.35
ax.bar([i - width/2 for i in x], period_summary['median'], width,
       label='Median', color='steelblue', edgecolor='white')
ax.bar([i + width/2 for i in x], period_summary['mean'], width,
       label='Mean', color='#f39c12', edgecolor='white')
ax.set_xticks(list(x))
ax.set_xticklabels(period_summary.index)
ax.set_ylabel('Monthly reviews')
ax.set_title('Average Monthly Review Volume by Pandemic Period', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(baseline_median, color='steelblue', linestyle=':', alpha=0.5, label='_nolegend_')
plt.tight_layout()
plt.show()

print(f"\nThe dotted line is the pre-COVID median ({baseline_median:.0f} reviews/month).")
print(f"Notice the gap between median and mean — that skew matters. A few high-volume tourist restaurants pull the mean up.")

In [ ]:
# ── Recovery trajectory — month-by-month as % of baseline ────────────────────
recovery_months = monthly_city[monthly_city['period'].isin(['Pre-COVID', 'Shock', 'Recovery'])].copy()
recovery_months['pct_of_baseline'] = (recovery_months['total_reviews'] / baseline_median) * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(recovery_months['review_month_dt'], recovery_months['pct_of_baseline'],
        color='steelblue', linewidth=1.5)
ax.axhline(100, color='gray', linestyle='--', alpha=0.6, label='Pre-COVID baseline (100%)')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-07-01'),
           alpha=0.15, color='red', label='Shock window')
ax.set_ylabel('% of pre-COVID median volume')
ax.set_xlabel('Month')
ax.set_title('Recovery Trajectory — Monthly Volume as Percentage of Pre-COVID Baseline',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Tourist vs. non-tourist pandemic impact ──────────────────────────────────
# Guardrail: ZIP 70130 dominates review counts. Split check is essential.

monthly_by_zone['period'] = monthly_by_zone['month_dt'].apply(assign_period)
zone_analysis = monthly_by_zone[monthly_by_zone['period'].isin(['Pre-COVID', 'Shock', 'Recovery'])]

zone_summary = (
    zone_analysis
    .groupby(['tourist_zone', 'period'])['review_count']
    .agg(['median', 'mean'])
    .reset_index()
)

for tz in [1, 0]:
    label = "Tourist zone" if tz == 1 else "Non-tourist"
    pre = zone_summary[(zone_summary['tourist_zone'] == tz) & (zone_summary['period'] == 'Pre-COVID')]
    shk = zone_summary[(zone_summary['tourist_zone'] == tz) & (zone_summary['period'] == 'Shock')]
    rec = zone_summary[(zone_summary['tourist_zone'] == tz) & (zone_summary['period'] == 'Recovery')]

    if not pre.empty and not shk.empty:
        base_med = pre['median'].values[0]
        shock_med = shk['median'].values[0]
        drop = (1 - shock_med / base_med) * 100 if base_med > 0 else 0
        rec_pct = (rec['median'].values[0] / base_med * 100) if not rec.empty and base_med > 0 else 0
        print(f"{label}:")
        print(f"  Pre-COVID median: {base_med:.0f} reviews/month")
        print(f"  Shock drop:       {drop:.1f}%")
        print(f"  Recovery:         {rec_pct:.1f}% of baseline")
        print()

## Pandemic Findings — First Draft

> *This section is supporting analysis — it answers the research question about business age dynamics and external shocks, but the project's primary focus remains predicting overall star rating (4.0 stars or above).*

### What we found

**Before COVID**, New Orleans restaurants collected a steady baseline of reviews each month. That rhythm had seasonal bumps — Mardi Gras, Jazz Fest — but the floor was reliable.

**During the shock window** (March–June 2020), review volume fell sharply. The drop was not uniform: tourist-zone restaurants, which depend on visitor traffic concentrated in a handful of ZIP codes, experienced a steeper decline than neighborhood restaurants whose customer base is more local.

**Recovery** has been uneven. City-wide volume began climbing back toward baseline, but the tourist zone and non-tourist areas recovered on different timelines. This matters because ZIP 70130 alone accounts for roughly 24% of all restaurants in the dataset — any city-wide average will be dominated by what happens there.

### What this does *not* tell us

- Review volume is a proxy for foot traffic, not a direct measure of revenue or survival.
- A restaurant that closed permanently during COVID would have dropped out of the "open" filter entirely — we only see survivors here.
- The recovery percentages compare volume to the pre-COVID baseline; they do not account for whether the reviews themselves became more positive or negative during recovery. That is Christian's territory (sentiment analysis).

### How this connects to the main project

The pandemic analysis lives here as context, not as a modeling input. The primary question remains: **which restaurant characteristics predict a rating of 4.0 stars or above?** The pandemic timeline helps explain *why* some patterns in the data look the way they do — particularly the review count skew in the tourist zone — but it does not change the feature set or the success label.

---

# Part 2 — Feature Engineering

**Owner: Kanniese**

Everything above built the cleaned, merged restaurant dataset with `success` and `tourist_zone` baked in. This section picks up that dataset and engineers the structured features the model will actually train on.

> **What changed from the standalone notebook:** The original `feature_engineering.ipynb` loaded a separate CSV (`new_orleans_restaurants.csv`) and re-read the raw business JSON. In this unified notebook, the data is already in memory — `merged` has the restaurant base, and `df_biz_full` has the raw business attributes. No file handoffs, no missing CSVs.

In [ ]:
# ── Feature Engineering: Step 1 — Connect to upstream data ──────────────────
# Instead of reading from a CSV that doesn't exist in the repo,
# use the merged dataset already in memory from Part 1.

import json as _json

restaurants = merged.copy()

# The full business JSON is already loaded as df_biz_full from Step 1.
# Filter to our restaurant set.
business_df = df_biz_full[df_biz_full["business_id"].isin(restaurants["business_id"])].copy()

print(f"Restaurants from merged dataset: {len(restaurants):,}")
print(f"Business records matched: {len(business_df):,}")

In [ ]:
# ── Feature Engineering: Step 2 — Extract business attributes ────────────────

def get_attribute(attr_dict, key):
    if isinstance(attr_dict, dict):
        return attr_dict.get(key)
    return None

# Price range
business_df["price_range"] = business_df["attributes"].apply(
    lambda x: get_attribute(x, "RestaurantsPriceRange2")
)

# Binary features
features = {
    "takeout": "RestaurantsTakeOut",
    "delivery": "RestaurantsDelivery",
    "reservations": "RestaurantsReservations",
    "outdoor_seating": "OutdoorSeating",
    "good_for_kids": "GoodForKids",
}

for new_col, attr_name in features.items():
    business_df[new_col] = business_df["attributes"].apply(
        lambda x: get_attribute(x, attr_name)
    )

print("Extracted attributes:")
for col in ["price_range"] + list(features.keys()):
    non_null = business_df[col].notna().sum()
    pct = non_null / len(business_df) * 100
    print(f"  {col}: {non_null:,} non-null ({pct:.1f}%)")

In [ ]:
# ── Feature Engineering: Step 3 — Clean binary features ──────────────────────

def convert_to_binary(val):
    if val in [True, "True", "true", 1]:
        return 1
    return 0

for col in features.keys():
    business_df[col] = business_df[col].apply(convert_to_binary)

# Convert price to numeric
business_df["price_range"] = pd.to_numeric(business_df["price_range"], errors="coerce")

print("Binary feature distributions:")
for col in features.keys():
    print(f"  {col}: {business_df[col].value_counts().to_dict()}")

In [ ]:
# ── Feature Engineering: Step 4 — Merge attributes onto restaurants ──────────

merged_feat = restaurants.merge(
    business_df[["business_id", "price_range"] + list(features.keys())],
    on="business_id",
    how="left",
)

print(f"Shape after attribute merge: {merged_feat.shape}")

In [ ]:
# ── Feature Engineering: Step 5 — Competition density ────────────────────────

competition = merged_feat.groupby("postal_code").size().reset_index(name="competition_density")
merged_feat = merged_feat.merge(competition, on="postal_code", how="left")

print("Competition density (top 5 ZIPs):")
display(competition.sort_values("competition_density", ascending=False).head())

In [ ]:
# ── Feature Engineering: Step 6 — Handle missing values ──────────────────────

# Fill price with median
merged_feat["price_range"] = merged_feat["price_range"].fillna(merged_feat["price_range"].median())

# Fill binary features with 0 (assume absent = not offered)
for col in features.keys():
    merged_feat[col] = merged_feat[col].fillna(0)

print(f"Missing values after imputation: {merged_feat[list(features.keys()) + ['price_range']].isna().sum().sum()}")

In [ ]:
# ── Feature Engineering: Step 7 — Final feature table ────────────────────────

feature_table = merged_feat[
    ["business_id", "stars", "review_count", "price_range",
     "takeout", "delivery", "reservations", "outdoor_seating",
     "good_for_kids", "postal_code", "competition_density",
     "success", "tourist_zone"]
].copy()

feature_table.to_csv("week2_feature_table.csv", index=False)

print(f"Feature table exported: week2_feature_table.csv")
print(f"Shape: {feature_table.shape}")
display(feature_table.head())

---

# Part 3 — Text Mining

**Owner: Christian**

This section runs sentiment analysis (VADER) and keyword extraction on the cleaned review text. The input is the `cleaned_reviews_with_text.csv` exported in Part 1, Step 10 — or use `df_reviews_full` which is already in memory.

**What Christian needs to add here:**
- VADER sentiment scoring on all reviews
- Business-level aggregation: `avg_sentiment`, `sentiment_variance`, `avg_review_length`
- Keyword comparison: TF-IDF for restaurants with 4.0 stars or above vs. below 3.5
- Sentiment by tourist zone
- Pandemic sentiment shift across the three periods (pre-COVID, shock, recovery)
- Export business-level sentiment CSV for Kanniese's modeling section

In [ ]:
# ── Text Mining: Christian's section ─────────────────────────────────────────
# df_reviews_full is already in memory from Part 1, Step 10.
# It has columns: business_id, stars, date, text
#
# Starter example:
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
# analyzer = SentimentIntensityAnalyzer()
# df_reviews_full['compound'] = df_reviews_full['text'].apply(
#     lambda t: analyzer.polarity_scores(t)['compound']
# )

print("Text mining section — Christian, your data is ready:")
print(f"  df_reviews_full: {len(df_reviews_full):,} reviews")
print(f"  Columns: {list(df_reviews_full.columns)}")
print(f"  Date range: {df_reviews_full['date'].min():%Y-%m-%d} to {df_reviews_full['date'].max():%Y-%m-%d}")

---

# Part 4 — Predictive Modeling

**Owner: Kanniese**

This section builds and compares predictive models using the feature table from Part 2. The `feature_table` dataframe is already in memory.

**What Kanniese needs to add here:**
- Linear Regression baseline (R², RMSE)
- Random Forest (R², RMSE, feature importance)
- Test whether text features from Christian improve R² beyond structured features alone
- Classification: Random Forest on binary `success` (4.0+ = 1, below 3.5 = 0) — accuracy, F1
- Newer vs. established business comparison (first review 2018+ vs. before 2018)
- Export `new_orleans_feature_matrix.csv` with all final features

In [ ]:
# ── Modeling: Kanniese's section ─────────────────────────────────────────────
# feature_table is already in memory from Part 2.
# Once Christian's sentiment CSV is merged in, add those columns too.
#
# Starter example:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, f1_score
#
# X = feature_table.drop(columns=['business_id', 'success', 'postal_code'])
# y = feature_table['success']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Modeling section — Kanniese, your feature table is ready:")
print(f"  feature_table: {len(feature_table):,} rows x {feature_table.shape[1]} columns")
print(f"  Columns: {list(feature_table.columns)}")
print(f"  Success distribution: {feature_table['success'].value_counts().to_dict()}")

---

# Part 5 — Visualizations

**Owner: Bobby**

These charts use the `feature_table` dataframe already built in Part 2. No CSV loading needed — the data flows directly from the feature engineering section above.

> **What changed from the standalone notebook:** The original `DSBA_6211_Group_Project_Visualizations.ipynb` loaded `week2_feature_table.csv` from a hardcoded Colab path. In this unified notebook, `feature_table` is already in memory. The `tourist_zone` column is also available directly — no need to approximate it from competition density.

In [ ]:
# ── Visualizations: Setup ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

# Use the feature_table already in memory from Part 2
df_viz = feature_table.copy()
df_viz["postal_code"] = df_viz["postal_code"].astype(str).str.split(".").str[0].str.zfill(5)

print(f"Visualization dataset: {len(df_viz):,} rows")
display(df_viz.head())

In [ ]:
# ── Ratings Distribution ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
sns.histplot(data=df_viz, x="stars", bins=10, kde=True)
plt.title("Ratings Distribution")
plt.xlabel("Star Rating")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("fig_eda_ratings_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Review Count Distribution ────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
sns.histplot(data=df_viz, x="review_count", bins=30, kde=True)
plt.title("Review Count Distribution")
plt.xlabel("Number of Reviews")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("fig_eda_review_count.png", dpi=300, bbox_inches="tight")
plt.show()

# Guardrail: always report median alongside mean
print(f"Review count — Median: {df_viz['review_count'].median():.0f}, Mean: {df_viz['review_count'].mean():.0f}")
print(f"That gap is the skew. Top-reviewed restaurants cluster in the tourist zone.")

In [ ]:
# ── Restaurant Count by ZIP Code ─────────────────────────────────────────────
zip_counts = df_viz["postal_code"].value_counts().sort_values(ascending=False).head(15).reset_index()
zip_counts.columns = ["postal_code", "restaurant_count"]

plt.figure(figsize=(12, 6))
sns.barplot(data=zip_counts, x="postal_code", y="restaurant_count")
plt.title("Top 15 Restaurant Counts by ZIP Code")
plt.xlabel("ZIP Code")
plt.ylabel("Number of Restaurants")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("fig_eda_zip_counts.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Tourist Zone vs. Non-Tourist Zone ────────────────────────────────────────
# Uses the actual tourist_zone variable (defined by ZIP code in Part 1, Step 7)
# instead of approximating from competition density.

zone_counts = (
    df_viz["tourist_zone"]
    .value_counts()
    .rename({1: "Tourist Zone", 0: "Non-Tourist"})
    .reset_index()
)
zone_counts.columns = ["zone_type", "restaurant_count"]

plt.figure(figsize=(8, 6))
sns.barplot(data=zone_counts, x="zone_type", y="restaurant_count",
            palette=["#f39c12", "#3498db"])
plt.title("Tourist Zone vs. Non-Tourist Zone Comparison")
plt.xlabel("Zone Type")
plt.ylabel("Number of Restaurants")
plt.tight_layout()
plt.savefig("fig_eda_tourist_zone.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Dashboard: All Four Charts ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.histplot(data=df_viz, x="stars", bins=10, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Ratings Distribution")
axes[0, 0].set_xlabel("Star Rating")

sns.histplot(data=df_viz, x="review_count", bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Review Count Distribution")
axes[0, 1].set_xlabel("Number of Reviews")

sns.barplot(data=zip_counts, x="postal_code", y="restaurant_count", ax=axes[1, 0])
axes[1, 0].set_title("Top 15 Restaurant Counts by ZIP Code")
axes[1, 0].tick_params(axis="x", rotation=45)

sns.barplot(data=zone_counts, x="zone_type", y="restaurant_count",
            palette=["#f39c12", "#3498db"], ax=axes[1, 1])
axes[1, 1].set_title("Tourist Zone vs. Non-Tourist Zone")

plt.tight_layout()
plt.savefig("fig_dashboard.png", dpi=300, bbox_inches="tight")
plt.show()

### Visualization Takeaways

- Restaurant ratings concentrate in the 3.5–4.5 star range, making it harder to differentiate on rating alone.
- Review counts are heavily right-skewed (median 22, mean ~100). A small set of tourist-zone restaurants capture a disproportionate share of engagement.
- ZIP 70130 dominates restaurant density — roughly 24% of the dataset sits in that single code.
- The tourist vs. non-tourist comparison uses the actual ZIP-based definition (70112, 70113, 70116, 70117, 70130), not an approximation.